# Extract DEX data from Dune — for ONE chosen regime window

Pull swap-level prices, per-swap gas, mint/burn liquidity, chain gas, and 1-minute USD prices for the
pair over a **single volatility-regime window** (from `study_dates.csv`, produced by
`market_regime_detection`), then save one folder of CSVs under `S.data_dir`.

- The regime is `STUDY.active_regime` (`"low"` / `"mid"` / `"high"`). Extraction starts
  `S.extract_lead_hours` (default 4h) *before* the window so price/liquidity are live and the
  block-level MEV/frequency EWMAs have converged by the first study block.
- Outputs go to a **per-regime folder** `<regime>_vol/data_analysis/` (auto-created), so extracting
  another regime never overwrites this one — just change `STUDY.active_regime` and re-run the whole
  pipeline. `study_dates.csv` is shared at the pair level (regime-independent). Downstream
  (`transform` …) reads the same `active_regime`, so the window stays consistent.

Install deps with `pip install -r requirements.txt`.

In [1]:
import os

from dotenv import load_dotenv

from arblib import config, data_io, dune_api, regime
from arblib.config import STUDY as S
from arblib import kraken_api

load_dotenv()
headers = dune_api.make_headers(os.environ["DUNE_API_KEY"])

REGIME = S.active_regime             # set in arblib.config: "low" | "mid" | "high"
if S.test_mode:
    win = regime.custom_window(S, S.test_start, S.test_end)
    print(f"[TEST MODE] extracting custom window {win['start_ts']} -> {win['end_ts']} "
          f"(explicit dates, no regime)")
else:
    win = regime.study_window(S, REGIME)
    print(f"Extracting regime={REGIME!r}: study {win['study_start']} -> {win['end_ts']}"
          f"  (Dune from {win['start_ts']}, {S.extract_lead_hours}h warm-up lead)")
params = win["collection_params"]
params

Extracting regime='low': study 2026-05-12 00:00:00 -> 2026-05-18 00:00:00  (Dune from 2026-05-11 20:00:00, 4h warm-up lead)


{'start_ts': '2026-05-11 20:00:00',
 'end_ts': '2026-05-18 00:00:00',
 'token0': '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2',
 'token1': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48',
 'chain': 'ethereum'}

## Swaps

One swap query per DEX, each left-joined with that DEX's per-swap gas on
block / pool / tx / event index, saved under `S.swaps_dir`.

In [2]:
merge_keys = ["evt_block_number", "pool"]   # aggregated swaps & gas join on (pool, block)


df_uniswap_swap = dune_api.run_dune_saved_query(config.SWAP_QUERY_IDS["uniswap"], params, headers, "Uniswap")
df_gas_uniswap = dune_api.run_dune_saved_query(config.GAS_QUERY_IDS["uniswap_gas_per_swap"], params, headers, "Gas uniswap")
if not df_uniswap_swap.empty and not df_gas_uniswap.empty:
    df_uniswap_swap = (df_uniswap_swap.merge(df_gas_uniswap, on=merge_keys, how="left")
                       .sort_values("evt_block_number").reset_index(drop=True))

[Uniswap] EXECUTE RESPONSE: {'execution_id': '01M0JR7S8YN25F0ECP4W4VJ5W5', 'state': 'QUERY_STATE_PENDING'}
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_COMPLETED
[Gas uniswap] EXECUTE RESPONSE: {'execution_id': '01M0JR92CBBWN8XBDJ5RRJWNK6', 'state': 'QUERY_STATE_PENDING'}
[Gas uniswap] STATUS: QUERY_STATE_EXECUTING
[Gas uniswap] STATUS: QUERY_STATE_EXECUTING
[Gas uniswap] STATUS: QUERY_STATE_COMPLETED


In [3]:

df_pancake_swap = dune_api.run_dune_saved_query(config.SWAP_QUERY_IDS["pancake"], params, headers, "Pancake")
df_gas_pancake = dune_api.run_dune_saved_query(config.GAS_QUERY_IDS["pancake_gas_per_swap"], params, headers, "Gas pancake")
if not df_pancake_swap.empty and not df_gas_pancake.empty:
    df_pancake_swap = (df_pancake_swap.merge(df_gas_pancake, on=merge_keys, how="left")
                       .sort_values("evt_block_number").reset_index(drop=True))

[Pancake] EXECUTE RESPONSE: {'execution_id': '01M0JRA8ZVZ7RV4222M15060F9', 'state': 'QUERY_STATE_PENDING'}
[Pancake] STATUS: QUERY_STATE_EXECUTING
[Pancake] STATUS: QUERY_STATE_EXECUTING
[Pancake] STATUS: QUERY_STATE_COMPLETED
[Gas pancake] EXECUTE RESPONSE: {'execution_id': '01M0JRAWJX63Y4SV4X8VCWSTMG', 'state': 'QUERY_STATE_PENDING'}
[Gas pancake] STATUS: QUERY_STATE_EXECUTING
[Gas pancake] STATUS: QUERY_STATE_COMPLETED


In [4]:


data_io.save_dataframes(
    {config.SWAP_FILES["df_uniswap"]: df_uniswap_swap, config.SWAP_FILES["df_pancake"]: df_pancake_swap},
    S.swaps_dir,
)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/low_vol/data_analysis/swaps/df_uniswap_swap.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/low_vol/data_analysis/swaps/df_pancake_swap.csv
Done.


## Liquidity

Mint / burn events per pool, one query per DEX, saved under `S.liquidity_dir`.

In [5]:
df_uniswap_liq = dune_api.run_dune_saved_query(config.LIQUIDITY_QUERY_IDS["uniswap"], params, headers, "Uniswap liquidity")
df_pancake_liq = dune_api.run_dune_saved_query(config.LIQUIDITY_QUERY_IDS["pancake"], params, headers, "Pancake liquidity")
if not df_uniswap_liq.empty:
    df_uniswap_liq = df_uniswap_liq.sort_values("evt_block_number").reset_index(drop=True)
if not df_pancake_liq.empty:
    df_pancake_liq = df_pancake_liq.sort_values("evt_block_number").reset_index(drop=True)

data_io.save_dataframes(
    {config.LIQUIDITY_FILES["df_uniswap"]: df_uniswap_liq, config.LIQUIDITY_FILES["df_pancake"]: df_pancake_liq},
    S.liquidity_dir,
)

[Uniswap liquidity] EXECUTE RESPONSE: {'execution_id': '01M0JRBAF0P9APW1MDS4Z5A9QW', 'state': 'QUERY_STATE_PENDING'}
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_COMPLETED
[Pancake liquidity] EXECUTE RESPONSE: {'execution_id': '01M0JRDCEAMQDWP6TKA1KD6RBR', 'state': 'QUERY_STATE_PENDING'}
[Pancake liquidi

## Chain gas

Per-block base fee + utilization for the whole chain, saved under `S.gas_dir`.

In [6]:
df_gas_chain = dune_api.run_dune_saved_query(config.GAS_QUERY_IDS["chain_gas_price"], params, headers, "Gas price")
if not df_gas_chain.empty:
    df_gas_chain = df_gas_chain.sort_values("block_number").reset_index(drop=True)

data_io.save_dataframes({config.GAS_FILES["chain_gas_price"]: df_gas_chain}, S.gas_dir)

[Gas price] dropping params not used by query 7748900: ['token0', 'token1']
[Gas price] EXECUTE RESPONSE: {'execution_id': '01M0JRDPC20NH92Z3A9ZZF8YGN', 'state': 'QUERY_STATE_PENDING'}
[Gas price] STATUS: QUERY_STATE_EXECUTING
[Gas price] STATUS: QUERY_STATE_COMPLETED
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/low_vol/data_analysis/gas/chain_gas_price.csv
Done.


## Kraken USD prices for X and Y

1-minute USD close prices for **X** (`token0`) and **Y** (`token1`) from the Kraken **Trades**
endpoint over the same window, saved under `S.prices_dir`. The fetch starts at `win["cex_start_ts"]`
(the extraction start widened by one averaging window) so the earliest swap has a full trailing
window of *past* CEX prices.

In [7]:
x_prices = kraken_api.usd_prices_1min(config.KRAKEN_PAIRS[S.token0.symbol], win["cex_start_ts"], win["end_ts"])
y_prices = kraken_api.usd_prices_1min(config.KRAKEN_PAIRS[S.token1.symbol], win["cex_start_ts"], win["end_ts"])

data_io.save_dataframes({S.x_price_path.name: x_prices, S.y_price_path.name: y_prices}, S.prices_dir)

USDCUSD: 108807 trades -> 8940 1-min bars (2026-05-11 19:00:00+00:00 -> 2026-05-17 23:59:00+00:00)
ETHUSD: 127152 trades -> 8940 1-min bars (2026-05-11 19:00:00+00:00 -> 2026-05-17 23:59:00+00:00)
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/low_vol/data_analysis/prices/X_USD_prices.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/low_vol/data_analysis/prices/Y_USD_prices.csv
Done.
